In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

In [ ]:
device = torch.device("mps")

In [ ]:
words = open('names.txt').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoio = {s:i+1 for i,s in enumerate(chars)}
stoio['.'] = 0
itos = {i:s for s, i in stoio.items()}

In [ ]:
block_size = 8

def build_dataset(words):
    X = []
    Y = []
    for word in words:
        context = [0] * block_size
        for ch in word + '.':
            ix = stoio[ch]
            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.shuffle(words)

n1 = int(.8 * len(words))
n2 = int(.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
n_hidden = 300
n_emb = 100
n_vocab = 27

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((n_vocab, n_emb), generator=g)

W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * (5/3) / (n_emb * block_size) ** .5

W2 = torch.randn((n_hidden, n_vocab)) * .01
b2 = torch.randn(n_vocab, generator=g) * 0

bngain = torch.ones(1, n_hidden)
bnbias = torch.zeros(1, n_hidden)

bnmean = torch.zeros(1, n_hidden)
bnstd = torch.ones(1, n_hidden)

Xtr, Ytr, Xdev, Ydev, Xte, Yte, C, W1, W2, b2, bngain, bnbias = [
    p.to(device)
    for p in [Xtr, Ytr, Xdev, Ydev, Xte, Yte, C, W1, W2, b2, bngain, bnbias]
]

params = [W1, W2, b2, C, bngain, bnbias]

for p in params:
    p.requires_grad = True

In [ ]:
lossi = []

In [ ]:
max_steps = 100000
batch_size = 50
t = time.perf_counter()
for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), device=device)
    emb = C[Xtr[ix]]
    hpreact = emb.view(-1, n_emb * block_size) @ W1
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact -bnmeani) / bnstdi + bnbias

    with torch.no_grad():
        bnmean = bnmean * .999 + bnmeani * .001
        bnstd = bnstd * .999 + bnstdi * .001

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    lossi.append(loss.item())

    for p in params:
        p.grad = None
    loss.backward()
    lr = .1 if i < max_steps / 2 else .01
    for p in params:
        p.data += -lr * p.grad

    if i % int(max_steps / 20) == 0:
        print(f'step {i}/{max_steps}: {loss.item():.4f}')

t = time.perf_counter() - t

In [ ]:
plt.imshow(h.abs() > .99, )

In [ ]:
with torch.no_grad():
    emb = C[Xtr]
    hpreact = emb.view(-1, n_emb * block_size) @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)

print(bnmean, bnstd)

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'dev': (Xdev, Ydev),
        'test': (Xte, Yte)
    }[split]
    emb = C[x]
    hpreact = (emb.view(-1, block_size * n_emb) @ W1)
    hpreact = bngain * (hpreact - bnmean) / bnstd + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return loss.item()

print(f'{t:.2f}s')
print(split_loss('train'))
print(split_loss('dev'))

With scaling
52.94s
1.9847157001495361
2.1042821407318115

Without scaling
53.38s
2.0430736541748047
2.1773693561553955

with kaiming
60.37s
2.015230178833008
2.0852794647216797

with bn
85.69s
2.0100855827331543
2.07128643989563

In [ ]:
((hpreact) / hpreact.std()).std()

In [ ]:
for _ in range(10):
    x = torch.tensor([stoio['s'], stoio['a'], stoio['k']])
    w = ''
    while True:
        e = C[x].view(-1, n_emb * block_size)
        hpreact = e @ W1
        hpreact = bngain * (hpreact - bnmean) / bnstd + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        p = logits.exp()
        p /= p.sum(1, keepdim=True)
        y = torch.multinomial(p, 1).item()
        if y == 0:
            break
        w += itos[y]
        x = torch.cat((x[1:], torch.tensor([y])), dim=0)
        # print(x)
        # break
    print('sak' + w)

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True, scale=False, kaiming=False):
        self.w = torch.randn((fan_in, fan_out))
        if scale:
            self.w *= .2
        if kaiming:
            self.w *= (5/3) / (fan_in) ** .5
        self.b = torch.randn(fan_out) if bias else None

    def __call__(self, x):
        out = x @ self.w
        if self.b != None:
            out = out + self.b
        return out

    def parameters(self):
        params = [self.w]
        if self.b != None:
            params.append(self.b)
        return params


class Emb:
    def __init__(self, dim, fan_out):
        self.fan_out = fan_out
        self.emb = torch.randn(dim)

    def parameters(self):
        return [self.emb]

    def __call__(self, x):
        return self.emb[x]


class FlattenConsecutive:
    def __init__(self, n):
        self.n = n

    def __call__(self, x: torch.Tensor):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        return x

    def parameters(self):
        return []


class Tanh:
    def __call__(self, x):
        return torch.tanh(x)

    def parameters(self):
        return []


class BatchNorm1d:
    def __init__(self, fan_in, momentum=.01):
        self.gain = torch.ones(fan_in)
        self.bias = torch.zeros(fan_in)
        self.running_mean = torch.zeros(fan_in)
        self.running_var = torch.ones(fan_in)
        self.momentum = momentum

    def __call__(self, x: torch.Tensor):
        if x.ndim == 2:
            dim = 0
        else:
            dim = (0, 1)
        if self.training:
            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True)

            with torch.no_grad():
                self.running_mean = self.running_mean * (1-self.momentum) + xmean * self.momentum
                self.running_var = self.running_var * (1-self.momentum) + xvar * self.momentum
        else:
            xmean = self.running_mean
            xvar = self.running_var
        return self.gain * ((x - xmean) / torch.sqrt(xvar + 1e-9) ) + self.bias

    def parameters(self):
        return [self.gain, self.bias]


class Container:
    def __init__(self, layers):
        self.layers = layers

        for p in self.parameters():
            p.requires_grad = True

    def __call__(self, x):
        out = x
        for l in self.layers:
            out = l(out)

        return out

    def parameters(self) -> list[torch.Tensor]:
        out = []
        for l in self.layers:
            out = out + l.parameters()
        return out


    def zero_grad(self):
        for p in self.parameters():
            p.grad = None

In [ ]:
n_hidden = 128
n_emb = 24
n_vocab = 27

model = Container([
    Emb((n_vocab, n_emb), n_emb * block_size),
    FlattenConsecutive(2), Linear(n_emb * 2, n_hidden, kaiming=True), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, kaiming=True), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, kaiming=True), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_vocab, scale=True)
])

print(sum(p.nelement() for p in model.parameters()))

In [ ]:
for p in model.layers:
    p.training = True

lossi = []
max_steps = 200000
for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (32, ))

    logits = model(Xtr[ix])
    loss = F.cross_entropy(logits, Ytr[ix])

    model.zero_grad()
    loss.backward()

    for p in model.parameters():
        lr = .1 if i < max_steps/2 else .01
        if i > max_steps * .75:
            lr = .001
        p.data += -1 * p.grad * lr

    if i % 10000 == 0:
        print(loss.item())

    lossi.append(loss.item())
    # break

plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))

In [ ]:
for p in model.layers:
    p.training = False

logits = model(Xdev)
loss = F.cross_entropy(logits, Ydev)
print(loss.item())

```
big net (10, 100): 2.1214065551757812
with kaiming: 2.1053993701934814
with batchnorm: 2.1008527278900146
with multistep lr: 2.095
with block size = 8: 2.0471999645233154
with block size = 8 (bn before thanh): 2.0335853099823
wavenet (68 hidden): 2.058053493499756
bigger wavenet (20, 100): 2.0047976970672607
bigger wavenet (24, 128)
```

In [ ]:
for _ in range(10):
    x = torch.tensor([[0, 0, 0, 0, 0, 0, 0, stoio['s']]])
    w = ''
    while True:
        logits = model(x)
        p = F.softmax(logits, dim=1)
        y = torch.multinomial(p, 1).item()
        if y == 0:
            break
        w += itos[y]
        x = torch.cat((x[:, 1:], torch.tensor([[y]])), dim=1)
        # print(x)
        # break
    print('s' + w)

In [ ]:
class FlattenConsecutive(torch.nn.Module):
    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        return x

class BatchNormLastDim(torch.nn.Module):
    def __init__(self, fan_in):
        super().__init__()
        self.bn = torch.nn.BatchNorm1d(fan_in)

    def forward(self, x):
        shape = x.shape
        x = self.bn(x.view(-1, shape[-1]))
        return x.view(shape)

In [ ]:
batch_size = 2048
max_steps = 12500

model = torch.nn.Sequential(
    torch.nn.Embedding(n_vocab, n_emb),
    FlattenConsecutive(2), torch.nn.Linear(
        2 * n_emb, n_hidden, bias=False), BatchNormLastDim(n_hidden), torch.nn.Tanh(),
    FlattenConsecutive(2), torch.nn.Linear(
        2 * n_hidden, n_hidden, bias=False), BatchNormLastDim(n_hidden), torch.nn.Tanh(),
    FlattenConsecutive(2), torch.nn.Linear(
        2 * n_hidden, n_hidden, bias=False), BatchNormLastDim(n_hidden), torch.nn.Tanh(),
    torch.nn.Linear(n_hidden, n_vocab)
)
optimizer = torch.optim.SGD(model.parameters(), lr=.5)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[int(max_steps*.25), int(max_steps*.50), int(max_steps*.75), int(max_steps*.90)], gamma=0.1)
model = model.to(device).train()

@torch.compile
def compute_loss(x, y):
    return F.cross_entropy(model(x), y)

lossi = []
t = time.perf_counter()
for step in range(max_steps):
    ix = torch.randint(Xtr.shape[0], (batch_size, ), device=Xtr.device)
    loss = compute_loss(Xtr[ix], Ytr[ix])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if step % (max_steps // 20) == 0:
        print(f'{step/max_steps:.2f}% | {time.perf_counter() - t:.2f}s -> {loss.item():.5f}')
    lossi.append(loss.detach())

plt.plot(torch.tensor(lossi).view(-1, (max_steps // 100)).mean(dim=1))

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xdev)
    loss = F.cross_entropy(logits, Ydev)
    print(loss.item())